In [1]:
import ultralytics
from ultralytics import YOLO
from PIL import Image, ImageDraw, ImageFont
import requests

import os
import cv2

from matplotlib import patches, text, patheffects
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import shutil
import yaml

ultralytics.checks()

Ultralytics 8.4.63 🚀 Python-3.13.1 torch-2.6.0 CPU (Apple M2 Pro)
Setup complete ✅ (10 CPUs, 16.0 GB RAM, 305.1/460.4 GB disk)


# Training
Only run this portion once!

In [5]:
model = YOLO("yolo26n.pt")
results = model.train(data="/Users/jonathanzhu/nematostella_videos/labeled_data/NematostellaTracking.yolo26/data.yaml", epochs=10, device='mps')

Ultralytics 8.4.63 🚀 Python-3.13.1 torch-2.6.0 MPS (Apple M2 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/jonathanzhu/nematostella_videos/labeled_data/NematostellaTracking.yolo26/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-4, nbs=64, nms=False, opset=None, optimize=False, opti

/opt/anaconda3/envs/nndl-env/lib/python3.13/site-packages/ultralytics/utils/tal.py:195: UserWarning: MPS: nonzero op is supported natively starting from macOS 14.0. Falling back on CPU. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/mps/operations/Indexing.mm:401.)
  bbox_scores[mask_gt] = pd_scores[ind[0], :, ind[1]][mask_gt]  # b, max_num_obj, h*w


       1/10      6.27G      1.447      1.738  0.0009913        252        640: 100% ━━━━━━━━━━━━ 315/315 1.7s/it 8:42<1.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 3.5s/it 1:582.7sss
                   all       1080      21249      0.902      0.815      0.942      0.678

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/10      7.09G      1.175     0.6848  0.0007773        232        640: 100% ━━━━━━━━━━━━ 315/315 1.9s/it 9:45<2.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 4.3s/it 2:253.1ss
                   all       1080      21249       0.96      0.941      0.988      0.757

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/10       7.9G      1.101     0.4759  0.0007225        240        640: 100% ━━━━━━━━━━━━ 315/315 1.8s/it 9:13<2.2s
                 Class     Ima

Loss Metrics: (lower being better)
- `box_loss` is a metric that refers to how well the model predicts positions. 
- `cls_loss` or classification loss refers to how well the model predicts what is in each box.
- `dfl_loss` or Distribution Focal Loss refers to how well the model is refining. 

Post-Epoch Metrics: (scaled from 0 to 1, with 1 being better)
- `P` refers to precision. It measures what percentage of the model's predictions are correct.
- `R` refers to recall. It measures what percentage of actual objects the model predicted.
- `mAP50` and `mAP50-95` are similar and refer to mean average precision, measuring Intersection over Union, where we see how much overlap there is on predicted boxes versus actual boxes. mAP50-95 is more strict and we should hope for a high value for this.

# Preliminary Results (numeric metrics)

In [4]:
best_model = YOLO("runs/detect/train-4/weights/best.pt")
train_results = best_model.val(data="/Users/jonathanzhu/nematostella_videos/labeled_data/NematostellaTracking.yolo26/data.yaml", device='mps', split='train')
val_results = best_model.val(data="/Users/jonathanzhu/nematostella_videos/labeled_data/NematostellaTracking.yolo26/data.yaml", device='mps', split='val')
test_results = best_model.val(data="/Users/jonathanzhu/nematostella_videos/labeled_data/NematostellaTracking.yolo26/data.yaml", device='mps', split='test')

Ultralytics 8.4.63 🚀 Python-3.13.1 torch-2.6.0 MPS (Apple M2 Pro)
YOLO26n summary (fused): 122 layers, 2,375,226 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.2±0.2 ms, read: 526.8±113.3 MB/s, size: 97.9 KB)
val: Scanning /Users/jonathanzhu/nematostella_videos/labeled_data/NematostellaTracking.yolo26/train/labels.cache... 5037 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5037/5037 391.2Mit/s 0.0s


/opt/anaconda3/envs/nndl-env/lib/python3.13/site-packages/ultralytics/utils/nms.py:95: UserWarning: MPS: nonzero op is supported natively starting from macOS 14.0. Falling back on CPU. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/mps/operations/Indexing.mm:401.)
  x = x[filt]


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 315/315 3.0it/s 1:46<0.3ss
                   all       5037      99642      0.983      0.988      0.994      0.817
               Planula       5037      75000      0.987      0.989      0.994      0.793
                 Polyp       2174      24642      0.979      0.987      0.994      0.841
Speed: 0.2ms preprocess, 5.2ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to /Users/jonathanzhu/Documents/GitHub/nematostellatracking/video_analysis/runs/detect/val
Ultralytics 8.4.63 🚀 Python-3.13.1 torch-2.6.0 MPS (Apple M2 Pro)
val: Fast image access ✅ (ping: 0.2±0.1 ms, read: 474.5±211.3 MB/s, size: 117.5 KB)
val: Scanning /Users/jonathanzhu/nematostella_videos/labeled_data/NematostellaTracking.yolo26/valid/labels.cache... 1080 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1080/1080 156.2Mit/s 0.0s
                 Class     Images  Instances      Box(P       

In [5]:
print("Class indices with average precision:", test_results.ap_class_index)
print("Average precision for all classes:", test_results.box.all_ap)
print("Average precision:", test_results.box.ap)
print("Average precision at IoU=0.50:", test_results.box.ap50)
print("Class indices for average precision:", test_results.box.ap_class_index)
print("Class-specific results:", test_results.box.class_result)
print("F1 score:", test_results.box.f1)
print("F1 score curve:", test_results.box.f1_curve)
print("Overall fitness score:", test_results.box.fitness)
print("Mean average precision:", test_results.box.map)
print("Mean average precision at IoU=0.50:", test_results.box.map50)
print("Mean average precision at IoU=0.75:", test_results.box.map75)
print("Mean average precision for different IoU thresholds:", test_results.box.maps)
print("Mean results for different metrics:", test_results.box.mean_results)
print("Mean precision:", test_results.box.mp)
print("Mean recall:", test_results.box.mr)
print("Precision:", test_results.box.p)
print("Precision curve:", test_results.box.p_curve)
print("Precision values:", test_results.box.prec_values)
print("Specific precision metrics:", test_results.box.px)
print("Recall:", test_results.box.r)
print("Recall curve:", test_results.box.r_curve)

Class indices with average precision: [0 1]
Average precision for all classes: [[    0.99415     0.99389     0.99295     0.99004     0.97503     0.94005     0.87031     0.72438      0.4228    0.032547]
 [      0.994     0.99392     0.99337     0.99211     0.98958     0.97749     0.96073     0.87774     0.54676    0.058056]]
Average precision: [    0.79361     0.83837]
Average precision at IoU=0.50: [    0.99415       0.994]
Class indices for average precision: [0 1]
Class-specific results: <bound method Metric.class_result of ultralytics.utils.metrics.Metric object with attributes:

all_ap: array([[    0.99415,     0.99389,     0.99295,     0.99004,     0.97503,     0.94005,     0.87031,     0.72438,      0.4228,    0.032547],
       [      0.994,     0.99392,     0.99337,     0.99211,     0.98958,     0.97749,     0.96073,     0.87774,     0.54676,    0.058056]])
ap: array([    0.79361,     0.83837])
ap50: array([    0.99415,       0.994])
ap_class_index: array([0, 1])
curves: []
curv

# Individual Results for an Image

In [2]:
#first go at visualizing results on an image
img_folders = os.listdir("/Users/jonathanzhu/nematostella_videos/imgs/")
img_folders
#results_vis = best_model(["/Users/jonathanzhu/nematostella_videos/imgs/wt_16C_3_dpf_03_27_2026_celldish/"])

['wt_25c_6_dpf_fert_04_10_2026',
 'rpa_bb_16C_9_dpf_celldish_04_02_2026',
 'rpa_bb_16C_3_dpf_movedtodish_03_27_2026',
 'rpa_aa_16C_3_dpf_fert_04_24_2026',
 'wt_25c_7_dpf_fert_04_10_2026',
 'wt_16C_2_dpf_03_26_2026',
 'rpa_bb_25C_8_dpf_celldish_04_01_2026',
 'rpa_aa_25C_3_dpf_03_27_2026_celldish',
 'rpa_bb_16C_3_dpf_fert_04_24_2026',
 'glw_ac_16c_3_dpf_fert_04_24_2026',
 'rpa_bb_16C_6_dpf_celldish_03_30_2026',
 'wt_25c_4_dpf_fert_04_10_2026',
 'wt_25C_6_dpf_celldish_03_30_2026',
 'rpa_aa_16C_3_dpf_movedtodish_03_27_2026',
 'wt_25c_5_dpf_fert_04_10_2026',
 'wt_16C_8_dpf_celldish_04_01_2026',
 'rpa_aa_16C_6_dpf_fert_04_24_2026',
 'wt_16C_3_dpf_03_27_2026_celldish',
 '.DS_Store',
 'rpa_bb_16C_2_dpf_03_26_2026',
 'rpa_aa_25C_8_dpf_celldish_04_01_2026',
 'rpa_aa_16C_7_dpf_fert_04_24_2026',
 'glw_ac_16c_5_dpf_fert_04_24_2026',
 'rpa_bb_25C_3_dpf_03_27_2026_celldish',
 'rpa_bb_16C_5_dpf_fert_04_24_2026',
 'rpa_aa_16C_6_dpf_celldish_03_30_2026',
 'rpa_bb_16C_6_dpf_fert_04_24_2026',
 'glw_ac_16c

In [3]:
images_list = os.listdir("/Users/jonathanzhu/nematostella_videos/imgs/" + img_folders[0])
images_list

['wt_25c_6_dpf_fert_04_10_20260158.png',
 'wt_25c_6_dpf_fert_04_10_20261246.png',
 'wt_25c_6_dpf_fert_04_10_20261520.png',
 'wt_25c_6_dpf_fert_04_10_20261534.png',
 'wt_25c_6_dpf_fert_04_10_20261252.png',
 'wt_25c_6_dpf_fert_04_10_20260164.png',
 'wt_25c_6_dpf_fert_04_10_20260602.png',
 'wt_25c_6_dpf_fert_04_10_20261508.png',
 'wt_25c_6_dpf_fert_04_10_20260616.png',
 'wt_25c_6_dpf_fert_04_10_20260170.png',
 'wt_25c_6_dpf_fert_04_10_20261285.png',
 'wt_25c_6_dpf_fert_04_10_20260825.png',
 'wt_25c_6_dpf_fert_04_10_20260831.png',
 'wt_25c_6_dpf_fert_04_10_20261291.png',
 'wt_25c_6_dpf_fert_04_10_20260819.png',
 'wt_25c_6_dpf_fert_04_10_20261722.png',
 'wt_25c_6_dpf_fert_04_10_20261044.png',
 'wt_25c_6_dpf_fert_04_10_20261050.png',
 'wt_25c_6_dpf_fert_04_10_20260428.png',
 'wt_25c_6_dpf_fert_04_10_20261736.png',
 'wt_25c_6_dpf_fert_04_10_20260400.png',
 'wt_25c_6_dpf_fert_04_10_20261078.png',
 'wt_25c_6_dpf_fert_04_10_20260366.png',
 'wt_25c_6_dpf_fert_04_10_20260372.png',
 'wt_25c_6_dpf_f

In [4]:
#if running this: restart the kernel and skip all the portions with best_model.val
#you will get the error of RuntimeError: Inference tensors do not track version counter

best_model = YOLO("runs/detect/train-4/weights/best.pt")
results_vis = best_model("/Users/jonathanzhu/nematostella_videos/imgs/" + img_folders[0] + "/" + images_list[0])
for r in results_vis:
    boxes = r.boxes  # Boxes object for bounding box outputs
    masks = r.masks  # Masks object for segmentation masks outputs
    keypoints = r.keypoints  # Keypoints object for pose outputs
    probs = r.probs  # Probs object for classification outputs
    obb = r.obb  # Oriented boxes object for OBB outputs
    r.show()  # display to screen
    r.save(filename="result_test.png")  # save to disk


image 1/1 /Users/jonathanzhu/nematostella_videos/imgs/wt_25c_6_dpf_fert_04_10_2026/wt_25c_6_dpf_fert_04_10_20260158.png: 640x640 14 Planulas, 9 Polyps, 61.7ms
Speed: 2.4ms preprocess, 61.7ms inference, 0.1ms postprocess per image at shape (1, 3, 640, 640)


In [5]:
boxes.data

tensor([[5.5759e+02, 6.6118e+02, 5.8070e+02, 6.7711e+02, 9.4271e-01, 1.0000e+00],
        [8.4869e+02, 7.0201e+02, 8.5754e+02, 7.1272e+02, 9.3240e-01, 0.0000e+00],
        [7.9783e+02, 6.4432e+02, 8.1944e+02, 6.6274e+02, 9.2697e-01, 1.0000e+00],
        [7.4826e+02, 7.0776e+02, 7.5814e+02, 7.1819e+02, 9.2407e-01, 0.0000e+00],
        [5.3611e+02, 3.5992e+02, 5.4676e+02, 3.7011e+02, 9.2207e-01, 0.0000e+00],
        [3.8803e+02, 3.2693e+02, 3.9944e+02, 3.3775e+02, 9.1080e-01, 0.0000e+00],
        [3.2859e+02, 1.6222e+02, 3.4463e+02, 1.7804e+02, 9.0965e-01, 1.0000e+00],
        [6.9980e+02, 1.4522e+02, 7.1358e+02, 1.7446e+02, 9.0280e-01, 1.0000e+00],
        [4.6722e+02, 1.6435e+02, 4.8033e+02, 1.7419e+02, 8.9996e-01, 0.0000e+00],
        [4.2034e+02, 5.1453e+02, 4.3001e+02, 5.2424e+02, 8.9826e-01, 0.0000e+00],
        [4.2429e+02, 5.7293e+02, 4.3173e+02, 5.8088e+02, 8.6077e-01, 0.0000e+00],
        [2.8675e+02, 3.9055e+02, 2.9953e+02, 4.0358e+02, 8.1966e-01, 0.0000e+00],
        [1.4450e

In [7]:
df_res = pd.DataFrame(np.array(boxes.data))
df_res

/var/folders/px/b7vc3nh913zb_m0x36ncftj00000gn/T/ipykernel_12103/1638245846.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  df_res = pd.DataFrame(np.array(boxes.data))


,0,1,2,3,4,5
0,557.586121,661.183105,580.698059,677.107971,0.942711,1.0
1,848.687317,702.012817,857.541931,712.724487,0.932403,0.0
2,797.833618,644.323486,819.435425,662.739746,0.926968,1.0
3,748.263428,707.756714,758.144043,718.186951,0.924072,0.0
4,536.110962,359.921844,546.759460,370.110138,0.922071,0.0
5,388.031128,326.925964,399.440491,337.754120,0.910799,0.0
6,328.585571,162.219955,344.634308,178.037628,0.909652,1.0
7,699.795959,145.218094,713.582642,174.458450,0.902802,1.0
8,467.221497,164.346786,480.329712,174.194046,0.899955,0.0
9,420.338226,514.527466,430.005035,524.239136,0.898264,0.0


In [8]:
df_res.columns = ["x1", "y1", "x2", "y2", "conf", "class"]
df_res

,x1,y1,x2,y2,conf,class
0,557.586121,661.183105,580.698059,677.107971,0.942711,1.0
1,848.687317,702.012817,857.541931,712.724487,0.932403,0.0
2,797.833618,644.323486,819.435425,662.739746,0.926968,1.0
3,748.263428,707.756714,758.144043,718.186951,0.924072,0.0
4,536.110962,359.921844,546.759460,370.110138,0.922071,0.0
5,388.031128,326.925964,399.440491,337.754120,0.910799,0.0
6,328.585571,162.219955,344.634308,178.037628,0.909652,1.0
7,699.795959,145.218094,713.582642,174.458450,0.902802,1.0
8,467.221497,164.346786,480.329712,174.194046,0.899955,0.0
9,420.338226,514.527466,430.005035,524.239136,0.898264,0.0


In [9]:
df_res.insert(0, "image_filename", images_list[0])

In [10]:
df_res

,image_filename,x1,y1,x2,y2,conf,class
0,wt_25c_6_dpf_fert_04_10_20260158.png,557.586121,661.183105,580.698059,677.107971,0.942711,1.0
1,wt_25c_6_dpf_fert_04_10_20260158.png,848.687317,702.012817,857.541931,712.724487,0.932403,0.0
2,wt_25c_6_dpf_fert_04_10_20260158.png,797.833618,644.323486,819.435425,662.739746,0.926968,1.0
3,wt_25c_6_dpf_fert_04_10_20260158.png,748.263428,707.756714,758.144043,718.186951,0.924072,0.0
4,wt_25c_6_dpf_fert_04_10_20260158.png,536.110962,359.921844,546.759460,370.110138,0.922071,0.0
5,wt_25c_6_dpf_fert_04_10_20260158.png,388.031128,326.925964,399.440491,337.754120,0.910799,0.0
6,wt_25c_6_dpf_fert_04_10_20260158.png,328.585571,162.219955,344.634308,178.037628,0.909652,1.0
7,wt_25c_6_dpf_fert_04_10_20260158.png,699.795959,145.218094,713.582642,174.458450,0.902802,1.0
8,wt_25c_6_dpf_fert_04_10_20260158.png,467.221497,164.346786,480.329712,174.194046,0.899955,0.0
9,wt_25c_6_dpf_fert_04_10_20260158.png,420.338226,514.527466,430.005035,524.239136,0.898264,0.0


In [11]:
df_res.to_csv("result_test.csv")